# BirdCLEF+ 2026 — Model Analysis

This notebook analyses trained model performance using out-of-fold (OOF) predictions.
OOF predictions are generated during cross-validation and represent unbiased estimates
of how the model generalises to unseen data.

**Sections**
1. Load OOF predictions
2. Overall OOF score
3. Per-class AUC analysis
4. Score distribution across folds
5. Hard species — where the model struggles
6. Calibration analysis
7. Prediction score distributions
8. Experiment comparison

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
sns.set_palette("muted")

# Paths
DATA_ROOT = Path("../data/raw")
TAXONOMY_CSV = DATA_ROOT / "taxonomy.csv"

# Point this to your experiment run directory after training
# Example: EXPERIMENT_DIR = Path("../experiments/perch_mlp_20260504_120000")
EXPERIMENT_DIR = Path("../experiments")

# Load taxonomy to get species labels
taxonomy = pd.read_csv(TAXONOMY_CSV)
species_list = taxonomy["primary_label"].tolist()
class_names = taxonomy.set_index("primary_label")["class_name"].to_dict()

print(f"Species: {len(species_list)}")
print(f"Taxonomic classes: {taxonomy['class_name'].unique().tolist()}")

## 2. Load OOF Predictions

In [ ]:
# Locate the most recent experiment run directory
# OOF predictions are saved automatically by Trainer.run()

def find_latest_run(experiments_dir: Path) -> Path:
    """Return the most recently created run directory."""
    runs = sorted(
        [d for d in experiments_dir.iterdir() if d.is_dir() and (d / "oof_preds.npy").exists()],
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    if not runs:
        raise FileNotFoundError(
            f"No completed run found in {experiments_dir}.\n"
            "Train a model first: python scripts/train.py"
        )
    return runs[0]

run_dir = find_latest_run(EXPERIMENT_DIR)
print(f"Loading from: {run_dir}")

oof_preds  = np.load(run_dir / "oof_preds.npy")   # shape: (n_samples, 234)
oof_labels = np.load(run_dir / "oof_labels.npy")  # shape: (n_samples, 234)

print(f"OOF predictions shape : {oof_preds.shape}")
print(f"OOF labels shape      : {oof_labels.shape}")
print(f"Prediction range      : [{oof_preds.min():.4f}, {oof_preds.max():.4f}]")
print(f"Positive rate (mean)  : {oof_labels.mean():.4f}")

## 3. Overall OOF Score

In [ ]:
import sys
sys.path.insert(0, "../src")
from birdclef.utils.metrics import macro_auc_skipping_empty, compute_oof_score

metrics = compute_oof_score(oof_preds, oof_labels)

print("=" * 45)
print("OOF RESULTS SUMMARY")
print("=" * 45)
print(f"  Macro AUC (competition metric) : {metrics['macro_auc']:.4f}")
print(f"  Classes scored                 : {metrics['n_classes_scored']}")
print(f"  Classes skipped (no positives) : {metrics['n_classes_skipped']}")
print(f"  Total classes                  : {oof_labels.shape[1]}")

## 4. Per-Class AUC Analysis

In [ ]:
from birdclef.utils.metrics import per_class_auc

# Compute AUC for each species independently
per_class_df = per_class_auc(oof_labels, oof_preds, species_list)

# Add taxonomic class
per_class_df["class_name"] = per_class_df["species"].map(class_names)

print(f"Species with AUC computed: {len(per_class_df)}")
print()
print(per_class_df.describe()[["n_positives", "auc"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# AUC distribution histogram
axes[0].hist(per_class_df["auc"], bins=40, color="steelblue", edgecolor="white")
axes[0].axvline(per_class_df["auc"].mean(), color="firebrick", linestyle="--",
                linewidth=1.5, label=f"Mean = {per_class_df['auc'].mean():.3f}")
axes[0].axvline(0.5, color="gray", linestyle=":", linewidth=1.5, label="Random (0.5)")
axes[0].set_xlabel("Per-class AUC")
axes[0].set_ylabel("Number of species")
axes[0].set_title("Distribution of per-class OOF AUC scores")
axes[0].legend()

# AUC by taxonomic class — boxplot
class_order = per_class_df.groupby("class_name")["auc"].median().sort_values(ascending=False).index
sns.boxplot(
    data=per_class_df, x="class_name", y="auc",
    order=class_order, ax=axes[1],
    palette="muted", linewidth=1.0
)
axes[1].axhline(0.5, color="gray", linestyle=":", linewidth=1.2)
axes[1].set_xlabel("Taxonomic class")
axes[1].set_ylabel("AUC")
axes[1].set_title("Per-class AUC by taxonomic group")

plt.tight_layout()
plt.savefig(run_dir / "per_class_auc_distribution.png", bbox_inches="tight")
plt.show()

# Summary table by class
print(per_class_df.groupby("class_name")["auc"].agg(["mean", "median", "min", "max"]).round(4))

## 5. Hard Species — Where the Model Struggles

In [ ]:
# Bottom 20 species by AUC — these are the hardest for the model
bottom20 = per_class_df.nsmallest(20, "auc")[["species", "class_name", "n_positives", "auc"]]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["firebrick" if auc < 0.5 else "steelblue" for auc in bottom20["auc"]]
bars = ax.barh(range(20), bottom20["auc"].values[::-1], color=colors[::-1])
ax.set_yticks(range(20))
ax.set_yticklabels(
    [f"{row.species} ({row.class_name}, n={row.n_positives})"
     for _, row in bottom20.iloc[::-1].iterrows()],
    fontsize=8
)
ax.axvline(0.5, color="gray", linestyle="--", linewidth=1.2, label="Random baseline")
ax.set_xlabel("AUC")
ax.set_title("Bottom 20 species by OOF AUC")
ax.legend()
plt.tight_layout()
plt.show()

print(bottom20.to_string(index=False))

In [ ]:
# Top 20 species — what the model is most confident on
top20 = per_class_df.nlargest(20, "auc")[["species", "class_name", "n_positives", "auc"]]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(20), top20["auc"].values[::-1], color="steelblue")
ax.set_yticks(range(20))
ax.set_yticklabels(
    [f"{row.species} ({row.class_name}, n={row.n_positives})"
     for _, row in top20.iloc[::-1].iterrows()],
    fontsize=8
)
ax.set_xlabel("AUC")
ax.set_title("Top 20 species by OOF AUC")
plt.tight_layout()
plt.show()

In [ ]:
# Relationship between number of training samples and AUC
# Expectation: more samples -> better AUC

fig, ax = plt.subplots(figsize=(10, 5))

for cls, color in [("Aves", "steelblue"), ("Amphibia", "coral"),
                   ("Insecta", "seagreen"), ("Mammalia", "mediumpurple"),
                   ("Reptilia", "goldenrod")]:
    subset = per_class_df[per_class_df["class_name"] == cls]
    if len(subset) == 0:
        continue
    ax.scatter(subset["n_positives"], subset["auc"],
               alpha=0.6, s=30, color=color, label=cls)

ax.axhline(0.5, color="gray", linestyle=":", linewidth=1.2, label="Random baseline")
ax.set_xscale("log")
ax.set_xlabel("Number of positive training examples (log scale)")
ax.set_ylabel("OOF AUC")
ax.set_title("AUC vs training sample count per species")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Correlation between sample count and AUC
corr = per_class_df[["n_positives", "auc"]].corr().iloc[0, 1]
print(f"Pearson correlation (n_positives vs AUC): {corr:.4f}")
print("Interpretation: stronger positive correlation means the model")
print("relies heavily on sample count — a sign that rare species need attention.")

## 6. Calibration Analysis

In [ ]:
# Calibration: do predicted probabilities match actual positive rates?
# A well-calibrated model has predicted prob ~= empirical positive rate.
# Poor calibration -> post-processing with temperature scaling helps.

# Flatten across all species with at least one positive
mask = oof_labels.sum(axis=0) > 0
flat_preds  = oof_preds[:, mask].ravel()
flat_labels = oof_labels[:, mask].ravel()

# Binarize labels at 0.5 threshold (primary label = 1.0, secondary = 0.5)
flat_labels_bin = (flat_labels >= 1.0).astype(int)

fraction_pos, mean_pred = calibration_curve(
    flat_labels_bin, flat_preds, n_bins=20, strategy="quantile"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration curve
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray",
             linewidth=1.2, label="Perfect calibration")
axes[0].plot(mean_pred, fraction_pos, marker="o", markersize=5,
             color="steelblue", linewidth=2, label="Model")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Fraction of positives")
axes[0].set_title("Reliability diagram (calibration curve)")
axes[0].legend()

# Prediction score histogram
axes[1].hist(flat_preds[flat_labels_bin == 0], bins=50, alpha=0.6,
             color="steelblue", label="Negatives", density=True)
axes[1].hist(flat_preds[flat_labels_bin == 1], bins=50, alpha=0.6,
             color="firebrick", label="Positives", density=True)
axes[1].set_xlabel("Predicted probability")
axes[1].set_ylabel("Density")
axes[1].set_title("Score distributions: positives vs negatives")
axes[1].legend()

plt.tight_layout()
plt.savefig(run_dir / "calibration_analysis.png", bbox_inches="tight")
plt.show()

In [ ]:
# Effect of temperature scaling on calibration
# Temperature T > 1 softens predictions toward 0.5 (reduces overconfidence)
# Temperature T < 1 sharpens predictions (increases confidence)

from scipy.special import expit as sigmoid

# Convert probabilities back to logits
eps = 1e-6
logits = np.log(flat_preds / (1 - flat_preds + eps) + eps)

temperatures = [0.5, 1.0, 1.5, 2.0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.2, label="Perfect")

for T in temperatures:
    scaled_preds = sigmoid(logits / T)
    fp, mp = calibration_curve(flat_labels_bin, scaled_preds,
                               n_bins=20, strategy="quantile")
    ax.plot(mp, fp, marker="o", markersize=4, linewidth=1.5, label=f"T = {T}")

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration curves at different temperature values")
ax.legend()
plt.tight_layout()
plt.show()

print("The temperature closest to the diagonal gives best calibration.")
print("Use this value as inference.temperature in configs/base_config.yaml")

## 7. ROC Curves for Selected Species

In [ ]:
# Plot ROC curves for top 5 and bottom 5 species by AUC
# This shows how well-separated the predictions are for good vs poor species

top5 = per_class_df.nlargest(5, "auc")["species"].tolist()
bot5 = per_class_df.nsmallest(5, "auc")["species"].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, species_set, title, color in [
    (axes[0], top5, "ROC curves — top 5 species (highest AUC)", "steelblue"),
    (axes[1], bot5, "ROC curves — bottom 5 species (lowest AUC)", "firebrick"),
]:
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.0)
    for sp in species_set:
        idx = species_list.index(sp)
        gt = (oof_labels[:, idx] >= 1.0).astype(int)
        if gt.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(gt, oof_preds[:, idx])
        auc = roc_auc_score(gt, oof_preds[:, idx])
        ax.plot(fpr, tpr, linewidth=1.5, alpha=0.8, label=f"{sp} ({auc:.3f})")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

## 8. Experiment Comparison

In [ ]:
# Load metrics from all available experiment runs for comparison
# Each run saves metrics.jsonl with per-epoch train_loss and val_auc

import json

def load_run_metrics(run_dir: Path) -> pd.DataFrame:
    """Load the metrics JSONL file for one experiment run."""
    metrics_file = run_dir / "metrics.jsonl"
    if not metrics_file.exists():
        return pd.DataFrame()
    records = []
    with open(metrics_file) as f:
        for line in f:
            try:
                records.append(json.loads(line.strip()))
            except json.JSONDecodeError:
                pass
    return pd.DataFrame(records)

# Load the current run
metrics_df = load_run_metrics(run_dir)

if len(metrics_df) > 0 and "epoch" in metrics_df.columns:
    epoch_df = metrics_df.dropna(subset=["epoch"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Training loss curve
    if "train_loss" in epoch_df.columns:
        axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"],
                     color="steelblue", linewidth=1.5, label="Train loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training loss over epochs")
        axes[0].legend()

    # Validation AUC curve
    if "val_auc" in epoch_df.columns:
        axes[1].plot(epoch_df["epoch"], epoch_df["val_auc"],
                     color="steelblue", linewidth=1.5, label="Val AUC")
        best_epoch = epoch_df.loc[epoch_df["val_auc"].idxmax()]
        axes[1].axvline(best_epoch["epoch"], color="firebrick", linestyle="--",
                        linewidth=1.2,
                        label=f"Best epoch {int(best_epoch['epoch'])} (AUC={best_epoch['val_auc']:.4f})")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Validation AUC")
        axes[1].set_title("Validation AUC over epochs")
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(run_dir / "training_curves.png", bbox_inches="tight")
    plt.show()
else:
    print("No epoch-level metrics found. Run training first.")

## 9. Key Findings

In [ ]:
print("MODEL ANALYSIS SUMMARY")
print("=" * 50)
print(f"  OOF macro AUC            : {metrics['macro_auc']:.4f}")
print(f"  Species scored           : {metrics['n_classes_scored']}")
print(f"  Median per-class AUC     : {per_class_df['auc'].median():.4f}")
print(f"  Species with AUC < 0.7   : {(per_class_df['auc'] < 0.7).sum()}")
print(f"  Species with AUC > 0.95  : {(per_class_df['auc'] > 0.95).sum()}")
print()

# Identify the most impactful improvement areas
low_auc_high_count = per_class_df[
    (per_class_df["auc"] < 0.8) & (per_class_df["n_positives"] >= 20)
]
print(f"  High-data species with AUC < 0.8: {len(low_auc_high_count)}")
print("  These are the highest-impact species to improve — they have")
print("  sufficient data but the model still struggles. Possible causes:")
print("    - Acoustic similarity to other species")
print("    - High intra-species variability in recordings")
print("    - Domain gap between XC/iNat and Pantanal PAM recordings")

if len(low_auc_high_count) > 0:
    print()
    print(low_auc_high_count.sort_values("auc")[["species", "class_name",
                                                   "n_positives", "auc"]].head(10).to_string(index=False))